# Etapa 2, E4 (osoba 2): overenie prostredia na datalabe

Notebook nič neinštaluje ani nesťahuje; iba zistí, čo je k dispozícii, aby sme vedeli vybrať LLM a spôsob inštalácie:

1. Python, CPU a RAM kontajnera
2. GPU podľa `nvidia-smi`
3. nainštalované balíčky (torch, transformers, bitsandbytes …)
4. či torch vidí GPU a krátky test rýchlosti vo float16
5. voľné miesto na disku (model 7B v 4-bit má asi 5 GB)
6. prístup na internet (pypi, huggingface, github)
7. či pôjde `pip install` a či je dostupný `git`
8. súhrn sa uloží do `overenie_datalab.json` vedľa notebooku

**Spustenie:** pri štarte servera zvoľ konfiguráciu **s GPU**, nahraj tento notebook, otvor ho a daj *Run → Run All Cells*. Trvá to do 1 minúty.

**Späť** prines súbor `overenie_datalab.json` (alebo skopíruj výpis poslednej bunky).

In [ ]:
# 1. Python, CPU, RAM
import json, os, platform, shutil, subprocess, sys, sysconfig, time

vysledok = {"cas": time.strftime("%Y-%m-%d %H:%M:%S")}
vysledok["python"] = sys.version.split()[0]
vysledok["python_cesta"] = sys.executable
vysledok["platforma"] = platform.platform()
vysledok["cpu_jadra"] = os.cpu_count()
try:
    with open("/proc/meminfo") as f:
        mem = dict(r.split(":", 1) for r in f)
    vysledok["ram_stroja"] = mem["MemTotal"].strip()
except Exception as e:
    vysledok["ram_stroja"] = f"nezistené ({e})"
# limit pamäte kontajnera (môže byť menší než RAM stroja)
for subor in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    if os.path.exists(subor):
        hodnota = open(subor).read().strip()
        vysledok["ram_limit_kontajnera"] = hodnota if not hodnota.isdigit() else f"{int(hodnota) / 1024**3:.1f} GB"
        break
vysledok["pracovny_priecinok"] = os.getcwd()
vysledok["home"] = os.path.expanduser("~")
for k, v in vysledok.items():
    print(f"{k:22} {v}")

In [ ]:
# 2. GPU podľa nvidia-smi
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout)
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version",
                        "--format=csv,noheader"], capture_output=True, text=True, timeout=30)
    vysledok["nvidia_smi"] = q.stdout.strip() or q.stderr.strip()
except FileNotFoundError:
    vysledok["nvidia_smi"] = "nvidia-smi nenájdené: asi je zvolená konfigurácia bez GPU"
print("súhrn:", vysledok["nvidia_smi"])

In [ ]:
# 3. Nainštalované balíčky ('-' = nie je nainštalovaný)
from importlib import metadata

balicky = ["torch", "transformers", "accelerate", "bitsandbytes", "huggingface_hub", "safetensors",
           "tokenizers", "pandas", "numpy", "scikit-learn", "pip"]
verzie = {}
for b in balicky:
    try:
        verzie[b] = metadata.version(b)
    except metadata.PackageNotFoundError:
        verzie[b] = "-"
    print(f"{b:16} {verzie[b]}")
vysledok["balicky"] = verzie

In [ ]:
# 4. Vidí torch GPU? Krátky test násobenia matíc vo float16
try:
    import torch
    vysledok["torch_cuda"] = torch.cuda.is_available()
    print("CUDA dostupná:", torch.cuda.is_available(), "| CUDA v torch:", torch.version.cuda)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        gpu = {"nazov": p.name, "pamat_GB": round(p.total_memory / 1024**3, 2),
               "compute_capability": f"{p.major}.{p.minor}", "cuda_v_torch": torch.version.cuda,
               "bf16_podporovane": torch.cuda.is_bf16_supported()}
        a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
        b = a @ a
        torch.cuda.synchronize()
        t = time.time()
        for _ in range(20):
            b = a @ a
        torch.cuda.synchronize()
        gpu["fp16_TFLOPS"] = round(20 * 2 * 4096**3 / (time.time() - t) / 1e12, 1)
        volna, celkova = torch.cuda.mem_get_info()
        gpu["volna_pamat_GB"] = round(volna / 1024**3, 2)
        del a, b
        torch.cuda.empty_cache()
        vysledok["gpu"] = gpu
        for k, v in gpu.items():
            print(f"{k:20} {v}")
except ImportError:
    vysledok["torch_cuda"] = "torch nie je nainštalovaný"
    print(vysledok["torch_cuda"])
except Exception as e:
    vysledok["torch_cuda"] = f"chyba: {e}"
    print(vysledok["torch_cuda"])

In [ ]:
# 5. Voľné miesto na disku a kam by sa ukladali modely
hf_cache = os.environ.get("HF_HOME") or os.path.join(os.path.expanduser("~"), ".cache", "huggingface")
vysledok["hf_cache"] = hf_cache
disk = {}
for meno, cesta in {"home": os.path.expanduser("~"), "pracovny": os.getcwd(), "tmp": "/tmp"}.items():
    try:
        u = shutil.disk_usage(cesta)
        disk[meno] = f"{cesta}: voľné {u.free / 1024**3:.1f} GB z {u.total / 1024**3:.1f} GB"
    except Exception as e:
        disk[meno] = f"{cesta}: nezistené ({e})"
    print(disk[meno])
vysledok["disk"] = disk
print("HF cache:", hf_cache)

In [ ]:
# 6. Prístup na internet
import urllib.request

adresy = {
    "pypi": "https://pypi.org/simple/bitsandbytes/",
    "huggingface": "https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct",
    "github": "https://github.com",
}
internet = {}
for meno, url in adresy.items():
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "python"}), timeout=15) as r:
            internet[meno] = f"OK ({r.status})"
    except Exception as e:
        internet[meno] = f"NEDOSTUPNÉ ({type(e).__name__}: {e})"
    print(f"{meno:12} {internet[meno]}")
vysledok["internet"] = internet
vysledok["proxy_nastavene"] = sorted(k for k in os.environ if "proxy" in k.lower())
print("proxy premenné:", vysledok["proxy_nastavene"] or "žiadne")

In [ ]:
# 7. Pôjde pip install? (len skúška, --dry-run nič nenainštaluje) a je dostupný git?
site = sysconfig.get_paths()["purelib"]
vysledok["site_packages_zapisovatelne"] = os.access(site, os.W_OK)
print("site-packages:", site, "| zapisovateľné:", vysledok["site_packages_zapisovatelne"])
try:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "bitsandbytes", "accelerate"],
                       capture_output=True, text=True, timeout=180)
    riadky = (r.stdout + r.stderr).strip().splitlines()
    vysledok["pip_dry_run"] = {"navratovy_kod": r.returncode, "posledne_riadky": riadky[-5:]}
    print("pip návratový kód:", r.returncode)
    print("\n".join(riadky[-5:]))
except Exception as e:
    vysledok["pip_dry_run"] = f"chyba: {e}"
    print(vysledok["pip_dry_run"])
vysledok["git"] = shutil.which("git") or "nie je"
print("git:", vysledok["git"])

In [ ]:
# 8. Súhrn: tento výpis alebo súbor overenie_datalab.json prines späť
with open("overenie_datalab.json", "w", encoding="utf-8") as f:
    json.dump(vysledok, f, ensure_ascii=False, indent=2)
print("uložené:", os.path.abspath("overenie_datalab.json"))
print(json.dumps(vysledok, ensure_ascii=False, indent=2))